# Week 42, Day 2: Spike-timing-dependent plasticity

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w042/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The window and its area

Write stdp(dt_ms) returning the weight change for a timing difference in milliseconds, and compute the window's area numerically over -200 to 200 ms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
Ap, Am, tp, tm = 0.005, 0.0065, 20.0, 20.0           # amplitudes and time constants (ms)

def stdp(dt_ms):
    ...

grid = np.linspace(-200, 200, 40001)
vals = np.array([stdp(d) for d in grid])
area = np.sum(vals) * (grid[1] - grid[0])
print(f"area {area:.4f} (A+ tau+ - A- tau- = {Ap * tp - Am * tm:.4f})")
plt.plot(grid, vals); plt.axhline(0, color="gray", lw=0.5); plt.xlim(-100, 100); plt.xlabel("t_post - t_pre (ms)"); plt.ylabel("weight change"); plt.show()

In [ ]:
import numpy as np
def test_window():
    "the window is positive for pre-before-post and negative for post-before-pre"
    assert stdp(10.0) > 0 and stdp(-10.0) < 0 and abs(stdp(10.0) - 0.005 * np.exp(-0.5)) < 1e-9

def test_area():
    "the numerical area matches A+ tau+ - A- tau- and is negative"
    assert abs(area - (Ap * tp - Am * tm)) < 1e-4 and area < 0

test_window()
test_area()
print("All checks passed")

### Exercise: Correlated inputs win

Complete the plasticity updates inside the loop: when inputs spike, depress their weights by Am times the postsynaptic trace (clipped at 0) and increase their traces by 1; when the neuron spikes, potentiate all weights by Ap times their traces (clipped at 1) and increase the postsynaptic trace by 1. (This simulates 100 s and takes a few seconds.)

In [ ]:
r = np.random.default_rng(2000)
dt, T, N, rate = 1e-3, 100.0, 100, 10.0
tau_m, V_th, V_reset, E_l, g = 0.02, -54e-3, -60e-3, -74e-3, 0.0016
w = np.full(N, 0.5); corr = np.arange(N) < 50
x_pre, x_post, V, n_spikes = np.zeros(N), 0.0, E_l, 0
decay_p, decay_m = np.exp(-1e3 * dt / tp), np.exp(-1e3 * dt / tm)
for k in range(int(T / dt)):
    common = r.random() < rate * dt                        # shared events for the correlated group
    pre = r.random(N) < rate * dt * 0.8
    pre[corr] |= common & (r.random(50) < 0.2)
    x_pre *= decay_p; x_post *= decay_m
    V += dt / tau_m * (E_l - V) + g * np.sum(w[pre])
    if pre.any():
        ...
    if V > V_th:
        V = V_reset; n_spikes += 1
        ...
rate_out = n_spikes / T
print(f"output {rate_out:.1f} Hz; correlated group {w[corr].mean():.2f}, uncorrelated group {w[~corr].mean():.2f}")
plt.hist([w[corr], w[~corr]], 20, label=["correlated", "uncorrelated"]); plt.xlabel("final weight"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_select():
    "the correlated group ends above 0.9 and the uncorrelated group below 0.3"
    assert w[corr].mean() > 0.9 and w[~corr].mean() < 0.3

def test_rate():
    "the output rate settles between 5 and 40 Hz"
    assert 5 < rate_out < 40

test_select()
test_rate()
print("All checks passed")

### Exercise: Latency reduction

For each repetition, integrate the 20 inputs in order (leaky, 20 ms membrane time constant) until the potential reaches the threshold of 4, record the spike time, and apply the STDP window to every input using its timing relative to the spike.

In [ ]:
times = np.arange(20) * 2.0                                  # ms
w_seq = np.full(20, 0.5); thr = 4.0; lat = []
A_lat_p, A_lat_m = 0.03, 0.035
for rep in range(150):
    V, t_spk = 0.0, None
    for i, t in enumerate(times):
        if i > 0:
            V *= np.exp(-(t - times[i - 1]) / 20.0)
        V += w_seq[i]
        if V >= thr:
            t_spk = t; break
    lat.append(t_spk)
    if t_spk is not None:
        d = t_spk - times
        ...
print(f"first spike at {lat[0]:.0f} ms; after 20 repetitions {lat[20]:.0f} ms; after 150 {lat[-1]:.0f} ms")
plt.plot(lat); plt.xlabel("repetition"); plt.ylabel("spike time in the sequence (ms)"); plt.show()

In [ ]:
import numpy as np
def test_earlier():
    "the output spike moves at least 15 ms earlier over 150 repetitions"
    assert lat[0] - lat[-1] >= 15

def test_early_inputs():
    "the first five inputs end stronger than the last five"
    assert w_seq[:5].mean() > w_seq[-5:].mean() + 0.5

test_earlier()
test_early_inputs()
print("All checks passed")